In [1]:
pip install PyPDF2 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install python-docx 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install chromadb 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
pip install transformers 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
pip install torch 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
pip install accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Imports
import docx
import PyPDF2
import os
import chromadb
from chromadb.utils import embedding_functions
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#  Document reading functions
def read_text_file(file_path: str) -> str:
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

def read_pdf_file(file_path: str) -> str:
    text = ""
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            text += page.extract_text() + "\n"
    return text

def read_docx_file(file_path: str) -> str:
    doc = docx.Document(file_path)
    return "\n".join([paragraph.text for paragraph in doc.paragraphs])

def read_document(file_path: str) -> str:
    _, ext = os.path.splitext(file_path)
    ext = ext.lower()
    if ext == '.txt':
        return read_text_file(file_path)
    elif ext == '.pdf':
        return read_pdf_file(file_path)
    elif ext == '.docx':
        return read_docx_file(file_path)
    else:
        raise ValueError(f"Unsupported format: {ext}")


In [ ]:
# Text chunking
def split_text(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """Split text into chunks with overlap"""
    # Split by Sanskrit and English delimiters
    sentences = []
    for delimiter in ['।', '.', '\n\n']:
        text = text.replace(delimiter, '|||')
    
    raw_sentences = [s.strip() for s in text.split('|||') if s.strip()]
    
    chunks = []
    current_chunk = []
    current_size = 0

    for sentence in raw_sentences:
        sent_size = len(sentence)
        
        if current_size + sent_size > chunk_size and current_chunk:
            chunks.append(' '.join(current_chunk))
            # Keep last sentence for overlap
            if overlap > 0 and len(current_chunk) > 1:
                current_chunk = [current_chunk[-1], sentence]
                current_size = len(current_chunk[-1]) + sent_size
            else:
                current_chunk = [sentence]
                current_size = sent_size
        else:
            current_chunk.append(sentence)
            current_size += sent_size

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

In [ ]:
# Initialize ChromaDB
print("Initializing ChromaDB...")
client = chromadb.PersistentClient(path="chroma_db")

sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

collection = client.get_or_create_collection(
    name="sanskrit_documents",
    embedding_function=sentence_transformer_ef
)

print(f"✓ Collection ready. Documents: {collection.count()}")


Initializing ChromaDB...
✓ Collection ready. Documents: 23


In [ ]:
#Document processing
def process_document(file_path: str) -> Tuple[List[str], List[str], List[Dict]]:
    """Process document into chunks"""
    try:
        print(f"Processing: {os.path.basename(file_path)}")
        content = read_document(file_path)
        chunks = split_text(content, chunk_size=500, overlap=50)
        
        file_name = os.path.basename(file_path)
        metadatas = [{"source": file_name, "chunk": i} for i in range(len(chunks))]
        ids = [f"{file_name}_chunk_{i}" for i in range(len(chunks))]
        
        print(f"  Created {len(chunks)} chunks")
        return ids, chunks, metadatas
    except Exception as e:
        print(f"Error: {e}")
        return [], [], []

def add_documents_to_collection(collection, folder_path: str):
    """Add all documents from folder"""
    if not os.path.exists(folder_path):
        print(f"❌ Folder not found: {folder_path}")
        return
    
    files = [os.path.join(folder_path, f) 
             for f in os.listdir(folder_path) 
             if os.path.isfile(os.path.join(folder_path, f))]
    
    print(f"\nFound {len(files)} files\n")
    
    for file_path in files:
        ids, texts, metadatas = process_document(file_path)
        if texts:
            # Add in batches
            batch_size = 100
            for i in range(0, len(texts), batch_size):
                end = min(i + batch_size, len(texts))
                collection.add(
                    documents=texts[i:end],
                    metadatas=metadatas[i:end],
                    ids=ids[i:end]
                )
            print(f"  ✓ Added to collection\n")

In [ ]:
#Load documents
folder_path = "data"
add_documents_to_collection(collection, folder_path)
print(f"\n{'='*60}")
print(f"Total documents in collection: {collection.count()}")
print('='*60)



Found 1 files

Processing: Rag-docs.docx
  Created 22 chunks
  ✓ Added to collection


Total documents in collection: 23


In [ ]:

def semantic_search(collection, query: str, n_results: int = 3) -> Dict:
    """Search for relevant documents"""
    return collection.query(
        query_texts=[query],
        n_results=n_results
    )

def extract_context(results) -> Tuple[str, List[str]]:
    """Extract context and sources"""
    context = "\n\n".join(results['documents'][0])
    sources = [
        f"{meta['source']} (chunk {meta['chunk']})" 
        for meta in results['metadatas'][0]
    ]
    return context, sources


In [ ]:
# Answer generation with lightweight approach (CPU-friendly)
import re
from collections import Counter

def generate_answer(query: str, context: str) -> str:
    """
    Smart answer generation without heavy models
    Works well on CPU for both English and Sanskrit
    """
    
    # Clean context
    context = context.replace('\n', ' ').strip()
    
    # Split into sentences intelligently
    sentences = split_into_sentences(context)
    
    # Determine question type
    question_type = detect_question_type(query)
    
    # Score and rank sentences
    ranked_sentences = rank_sentences(query, sentences, question_type)
    
    if not ranked_sentences:
        return "Information not found in the retrieved documents."
    
    # Build answer based on question type
    answer = build_answer(query, ranked_sentences, question_type)
    
    return answer

def split_into_sentences(text: str) -> list:
    """Split text into sentences (handles Sanskrit and English)"""
    # Replace Sanskrit and English sentence endings
    text = text.replace('।', '.')
    text = text.replace('॥', '.')
    
    # Split on periods, but keep the sentence
    sentences = []
    current = ""
    
    for char in text:
        current += char
        if char in ['.', '!', '?']:
            sent = current.strip()
            if len(sent) > 15:  # Ignore very short fragments
                sentences.append(sent)
            current = ""
    
    if current.strip() and len(current.strip()) > 15:
        sentences.append(current.strip())
    
    return sentences

def detect_question_type(query: str) -> str:
    """Detect what kind of question this is"""
    query_lower = query.lower()
    
    # Question word mappings
    if any(word in query_lower for word in ['who', 'कः', 'के']):
        return 'WHO'
    elif any(word in query_lower for word in ['what', 'किम्', 'क्या']):
        return 'WHAT'
    elif any(word in query_lower for word in ['when', 'कदा', 'कब']):
        return 'WHEN'
    elif any(word in query_lower for word in ['where', 'कुत्र', 'कहाँ']):
        return 'WHERE'
    elif any(word in query_lower for word in ['why', 'किमर्थम्', 'क्यों']):
        return 'WHY'
    elif any(word in query_lower for word in ['how', 'कथम्', 'कैसे']):
        return 'HOW'
    elif any(word in query_lower for word in ['tell', 'story', 'about', 'कथा', 'describe']):
        return 'STORY'
    else:
        return 'GENERAL'

def rank_sentences(query: str, sentences: list, question_type: str) -> list:
    """Rank sentences by relevance to query"""
    
    query_lower = query.lower()
    query_words = set(re.findall(r'\w+', query_lower))
    
    scored_sentences = []
    
    for sent in sentences:
        sent_lower = sent.lower()
        score = 0
        
        # Base score: word overlap
        sent_words = set(re.findall(r'\w+', sent_lower))
        common_words = query_words & sent_words
        score += len(common_words) * 2
        
        # Bonus points based on question type
        if question_type == 'WHO':
            if any(word in sent_lower for word in ['king', 'राजा', 'bhoj', 'भोज', 'kAlIdAsa', 'कालीदास']):
                score += 5
        
        elif question_type == 'STORY' or question_type == 'WHAT':
            if any(word in sent_lower for word in ['story', 'कथा', 'once', 'एकदा', 'clever', 'चतुर']):
                score += 5
            # Bonus for narrative indicators
            if any(word in sent_lower for word in ['poet', 'कवि', 'proclaimed', 'घोषित', 'court', 'दरबार']):
                score += 3
        
        elif question_type == 'WHEN':
            if any(word in sent_lower for word in ['once', 'एकदा', 'day', 'दिवस', 'time', 'काल']):
                score += 5
        
        elif question_type == 'WHY':
            if any(word in sent_lower for word in ['because', 'यतः', 'so', 'therefore', 'अतः']):
                score += 5
        
        # Penalize very short sentences
        if len(sent.split()) < 5:
            score -= 3
        
        # Penalize sentences that are just metadata or formatting
        if any(marker in sent for marker in ['by:', 'edu', '@', 'http']):
            score -= 10
        
        if score > 0:
            scored_sentences.append((score, sent))
    
    # Sort by score (highest first)
    scored_sentences.sort(reverse=True, key=lambda x: x[0])
    
    return scored_sentences

def build_answer(query: str, ranked_sentences: list, question_type: str) -> str:
    """Build final answer from ranked sentences"""
    
    if not ranked_sentences:
        return "No relevant information found."
    
    # For STORY type questions, give more context
    if question_type == 'STORY' or question_type == 'WHAT':
        # Take top 4-5 sentences to tell the story
        answer_parts = []
        for score, sent in ranked_sentences[:5]:
            # Clean up the sentence
            cleaned = clean_sentence(sent)
            if cleaned and len(cleaned) > 20:
                answer_parts.append(cleaned)
        
        if answer_parts:
            answer = " ".join(answer_parts[:3])  # Limit to 3 sentences
            # Add summary if too long
            if len(answer) > 600:
                answer = answer[:600] + "..."
            return answer
    
    # For WHO/WHEN/WHERE - more concise
    elif question_type in ['WHO', 'WHEN', 'WHERE']:
        # Take top 2 sentences
        answer_parts = []
        for score, sent in ranked_sentences[:2]:
            cleaned = clean_sentence(sent)
            if cleaned and len(cleaned) > 20:
                answer_parts.append(cleaned)
        
        if answer_parts:
            return " ".join(answer_parts)
    
    # For other questions
    else:
        # Take top 3 sentences
        answer_parts = []
        for score, sent in ranked_sentences[:3]:
            cleaned = clean_sentence(sent)
            if cleaned and len(cleaned) > 20:
                answer_parts.append(cleaned)
        
        if answer_parts:
            return " ".join(answer_parts[:2])
    
    # Fallback
    return clean_sentence(ranked_sentences[0][1])

def clean_sentence(sent: str) -> str:
    """Clean up a sentence for display"""
    # Remove extra spaces
    sent = re.sub(r'\s+', ' ', sent).strip()
    
    # Remove author attributions
    sent = re.sub(r'by:.*?@.*?(edu|com)', '', sent, flags=re.IGNORECASE)
    
    # Remove metadata markers
    sent = re.sub(r'\*\*.*?\*\*', '', sent)
    
    # Remove excessive punctuation
    sent = re.sub(r'[।.]{2,}', '.', sent)
    
    return sent.strip()


In [ ]:
#Complete RAG pipeline
def rag_query(query: str, n_results: int = 5, verbose: bool = True) -> Dict:
    """Complete RAG query pipeline"""
    
    if verbose:
        print("\n" + "="*60)
        print("QUERY PROCESSING")
        print("="*60)
        print(f"📝 Query: {query}\n")
    
    # Step 1: Retrieve
    if verbose:
        print("Step 1: Retrieving relevant documents...")
    results = semantic_search(collection, query, n_results=n_results)
    
    # Display retrieved docs
    if verbose:
        print("\n" + "="*60)
        print("RETRIEVED DOCUMENTS")
        print("="*60)
        for i in range(len(results['documents'][0])):
            doc = results['documents'][0][i]
            meta = results['metadatas'][0][i]
            score = 1 - results['distances'][0][i]
            print(f"\n📄 Document {i+1}")
            print(f"   Source: {meta['source']}, Chunk {meta['chunk']}")
            print(f"   Relevance: {score:.3f}")
            print(f"   Preview: {doc[:150]}...")
            print("-" * 60)
    
    # Step 2: Extract context
    context, sources = extract_context(results)
    
    # Step 3: Generate answer
    if verbose:
        print("\nStep 2: Generating answer from context...")
    answer = generate_answer(query, context)
    
    return {
        'query': query,
        'answer': answer,
        'context': context,
        'sources': sources,
        'num_sources': len(sources)
    }

def print_result(result: Dict):
    """Pretty print result"""
    print("\n" + "="*60)
    print("FINAL ANSWER")
    print("="*60)
    print(f"\n❓ Query: {result['query']}")
    print(f"\n💡 Answer:\n{result['answer']}")
    print(f"\n📚 Sources ({result['num_sources']}):")
    for i, source in enumerate(result['sources'], 1):
        print(f"   {i}. {source}")
    print("="*60)



In [ ]:
#Example query 1 - Sanskrit
print("\n\n" + "="*70)
print("EXAMPLE 1: Sanskrit Query")
print("="*70)

query1 = "देवः कदा साहाय्यं करोति ?"
result1 = rag_query(query1, n_results=3)
print_result(result1)



EXAMPLE 1: Sanskrit Query

QUERY PROCESSING
📝 Query: देवः कदा साहाय्यं करोति ?

Step 1: Retrieving relevant documents...

RETRIEVED DOCUMENTS

📄 Document 1
   Source: Rag-docs.docx, Chunk 13
   Relevance: 0.672
   Preview: सः प्रतिदिने भक्त्या देवस्य प्रार्थनाम् करोति "देव, कृपया मह्यं आरोग्यम् ददातु, धनम् ददातु" इति सः किंचित् अपि प्रयत्नम् न करोति, कार्यम् न करोति देवः...
------------------------------------------------------------

📄 Document 2
   Source: Rag-docs.docx, Chunk 16
   Relevance: 0.547
   Preview: भवान् द्रष्टुम् अपि न आगतवान् किमर्थम् ?” तदा देवः उक्तवान् "भो भक्त, अहम् त्रिवारम् आगतवान् पृष्टवान् च 'साहाय्यम् अवश्यकम् वा ?” इति परंतु भवान् न स...
------------------------------------------------------------

📄 Document 3
   Source: Rag-docs.docx, Chunk 11
   Relevance: 0.394
   Preview: अकस्मादेव घण्टानादः अजायत् घण्टानादेन चकिताः ते पुनः पुनः घण्टामधुन्वन्, घण्टाणादं च अकुर्वन् चित्रपुरस्थाः नागरिकाः वारं वारं पर्वतशिखरप्रदेशात् घण्ट...
-----------------------------

In [65]:
# CELL 12: Example query 2 - English
print("\n\n" + "="*70)
print("EXAMPLE 2: English Query")
print("="*70)

query2 = "Tell me about the clever Kalidas story"
result2 = rag_query(query2, n_results=3)
print_result(result2)



EXAMPLE 2: English Query

QUERY PROCESSING
📝 Query: Tell me about the clever Kalidas story

Step 1: Retrieving relevant documents...

RETRIEVED DOCUMENTS

📄 Document 1
   Source: Rag-docs.docx, Chunk 3
   Relevance: 0.625
   Preview: मूर्खभृत्यस्य संसर्गात् सर्वम् कार्यम् विनश्यति ॥             चतुरस्य कालीदासस्य Of the clever kAlIdAsa by: Kedar Naphade,                            ...
------------------------------------------------------------

📄 Document 2
   Source: Rag-docs.docx, Chunk 10
   Relevance: 0.572
   Preview: None of the scholars were able to say that they knew this poetry So the poet got one lakh rupees kAlIdAsa was indeed clever!)                         ...
------------------------------------------------------------

📄 Document 3
   Source: Rag-docs.docx, Chunk 19
   Relevance: 0.504
   Preview: पालखीं स्कन्दयोः वहन् निर्गतः कालीदासः पण्डितेन सह तस्मिन् काले शिशिरः भवति ऋतुः, शीतः च पवनः देहं ताडयति इव वदति पण्डितः, शीतं बहु बाधति इति चतुरः का...
------------------

In [48]:
# CELL 13: Example query 3 - Another query
print("\n\n" + "="*70)
print("EXAMPLE 3: Another Query")
print("="*70)

query3 = "Who is Shankhanada, and whom does he serve?"
result3 = rag_query(query3, n_results=3)
print_result(result3)



EXAMPLE 3: Another Query

QUERY PROCESSING
📝 Query: Who is Shankhanada, and whom does he serve?

Step 1: Retrieving relevant documents...

RETRIEVED DOCUMENTS

📄 Document 1
   Source: Rag-docs.docx, Chunk 3
   Relevance: 0.388
   Preview: मूर्खभृत्यस्य संसर्गात् सर्वम् कार्यम् विनश्यति ॥             चतुरस्य कालीदासस्य Of the clever kAlIdAsa by: Kedar Naphade,                            ...
------------------------------------------------------------

📄 Document 2
   Source: Rag-docs.docx, Chunk 11
   Relevance: 0.369
   Preview: अकस्मादेव घण्टानादः अजायत् घण्टानादेन चकिताः ते पुनः पुनः घण्टामधुन्वन्, घण्टाणादं च अकुर्वन् चित्रपुरस्थाः नागरिकाः वारं वारं पर्वतशिखरप्रदेशात् घण्ट...
------------------------------------------------------------

📄 Document 3
   Source: Rag-docs.docx, Chunk 16
   Relevance: 0.342
   Preview: भवान् द्रष्टुम् अपि न आगतवान् किमर्थम् ?” तदा देवः उक्तवान् "भो भक्त, अहम् त्रिवारम् आगतवान् पृष्टवान् च 'साहाय्यम् अवश्यकम् वा ?” इति परंतु भवान् न स...
-------------

In [66]:
# CELL 14: Interactive query system
def interactive_mode():
    """Interactive RAG system"""
    print("\n" + "="*60)
    print("SANSKRIT RAG SYSTEM - INTERACTIVE MODE")
    print("="*60)
    print("\nCommands:")
    print("  - Enter your question")
    print("  - 'stats' - show collection statistics")
    print("  - 'quit' - exit")
    print("\n" + "="*60)
    
    while True:
        user_input = input("\n🔍 Your question: ").strip()
        
        if not user_input:
            continue
        
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\n✓ Thank you for using the Sanskrit RAG System!")
            break
        
        if user_input.lower() == 'stats':
            print(f"\n📊 Statistics:")
            print(f"   Total documents: {collection.count()}")
            print(f"   Collection: {collection.name}")
            continue
        
        # Process query
        result = rag_query(user_input, n_results=3, verbose=False)
        print_result(result)
interactive_mode()


SANSKRIT RAG SYSTEM - INTERACTIVE MODE

Commands:
  - Enter your question
  - 'stats' - show collection statistics
  - 'quit' - exit


FINAL ANSWER

❓ Query: शंखनादः कः अस्ति ?

💡 Answer:
अकस्मादेव घण्टानादः अजायत् घण्टानादेन चकिताः ते पुनः पुनः घण्टामधुन्वन्, घण्टाणादं च अकुर्वन् चित्रपुरस्थाः नागरिकाः वारं वारं पर्वतशिखरप्रदेशात् घण्टानादमाकर्णयन् भयाकुलाः ते अचिन्तयन् "नुनं शिखरप्रदेशे घण्टकर्णः नाम राक्षसः वर्तते, यः मनुष्यां खादति, घण्टां च वादयति” एवं च भीत्या पौरजनाः अन्यत्र गन्तुं प्रारभन्त तदा चिन्ताकुलः नृपः उदघोषयत् "यः घण्टकर्णं नाशयेत्, तस्मै विपुलं सुवर्णं यच्छेयम् अहम्" इति तत् श्रुत्वा काचन वृद्धा वनं गता मूर्खभृत्यस्य "अरे शंखनाद, गच्छापणम्, शर्कराम् आनय " इति स्वभृत्यम् शंखनादम् गोवर्धनदासः आदिशति ततः शंखनादः आपणम् गच्छति, शर्कराम् जीर्णे वस्त्रे न्यस्यति च तस्मात् जीर्णवस्त्रात् मार्गे एव सर्वापि शर्करा स्त्रवति ततः गोवर्धनदासः कोपेन शंखनादम् वदति, "अरे मूढ, कुत्रास्ति शर्करा ? शर्करादिकम् एवम् जीर्णेन वस्त्रेण न एवानयन्ति कदापि इतःपरम् किमपि वस्तुजातम् दृढायाम् सन्

In [ ]:
#Performance metrics
import time

def evaluate_system(test_queries: List[str]):
    """Evaluate system performance"""
    print("\n" + "="*60)
    print("PERFORMANCE EVALUATION")
    print("="*60)
    
    results = []
    total_time = 0
    
    for i, query in enumerate(test_queries, 1):
        print(f"\nQuery {i}/{len(test_queries)}: {query}")
        
        start = time.time()
        result = rag_query(query, n_results=3, verbose=False)
        elapsed = time.time() - start
        
        total_time += elapsed
        results.append({
            'query': query,
            'time': elapsed,
            'answer_length': len(result['answer']),
            'sources': result['num_sources']
        })
        
        print(f"  ⏱️  Time: {elapsed:.2f}s")
        print(f"  📄 Sources: {result['num_sources']}")
        print(f"  📝 Answer length: {len(result['answer'])} chars")
    
    # Summary
    avg_time = total_time / len(test_queries)
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"Total queries: {len(test_queries)}")
    print(f"Total time: {total_time:.2f}s")
    print(f"Average time: {avg_time:.2f}s per query")
    print(f"Collection size: {collection.count()} documents")
    print("="*60)
    
    return results

# Test queries
test_queries = [
    "देवः कदा साहाय्यं करोति ?",
    "Tell me about the clever Kalidas story",
    "What is the moral of the story?",
    "Who was King Bhoj?"
]

eval_results = evaluate_system(test_queries)


PERFORMANCE EVALUATION

Query 1/4: देवः कदा साहाय्यं करोति ?
  ⏱️  Time: 0.02s
  📄 Sources: 3
  📝 Answer length: 427 chars

Query 2/4: Tell me about the clever Kalidas story
  ⏱️  Time: 0.02s
  📄 Sources: 3
  📝 Answer length: 427 chars

Query 3/4: What is the moral of the story?
  ⏱️  Time: 0.02s
  📄 Sources: 3
  📝 Answer length: 500 chars

Query 4/4: Who was King Bhoj?
  ⏱️  Time: 0.02s
  📄 Sources: 3
  📝 Answer length: 427 chars

SUMMARY
Total queries: 4
Total time: 0.08s
Average time: 0.02s per query
Collection size: 23 documents


In [ ]:
#Export results
import json

def export_results(results: List[Dict], filename: str = "rag_results.json"):
    """Export results to JSON"""
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\n✓ Results saved to {filename}")

# Save evaluation results
export_data = []
for result in eval_results:
    export_data.append({
        'query': result['query'],
        'processing_time_seconds': result['time'],
        'answer_length': result['answer_length'],
        'sources_retrieved': result['sources']
    })

export_results(export_data, "evaluation_results.json")

print("\n✅ Sanskrit RAG System is ready!")
print("="*60)


✓ Results saved to evaluation_results.json

✅ Sanskrit RAG System is ready!
